# Exploratory Data Analysis

This notebook is the first step in the Seattle building energy case study. It introduces the raw dataset, inspects column structure, evaluates missing values, and reviews the semantic role of the main fields before any cleaning or modeling.

## Objectives

- Inspect the raw Seattle energy dataset.
- Identify key columns for prediction and feature engineering.
- Check missing data, categorical variables, and compliance metadata.
- Save a filtered dataset for use in the preparation notebook.

In [ ]:
import pandas as pd
import numpy as np

## Load the raw dataset

In [ ]:
raw_path = "data/raw/2016_Building_Energy_Benchmarking.csv"
df = pd.read_csv(raw_path)
print("Raw dataset loaded from:", raw_path)
print("Shape:", df.shape)
print("Columns:", len(df.columns))
df.columns.tolist()[:20]

## Select non-residential buildings

The goal is to narrow the dataset to building types that are relevant for energy consumption modeling in non-residential and large campus settings.

In [ ]:
non_residential_keywords = [
    "NonResidential",
    "Nonresidential COS",
    "Nonresidential WA",
    "SPS-District K-12",
    "Campus"
]

candidate_columns = [
    "BuildingType",
    "PrimaryPropertyType",
    "LargestPropertyUseType",
    "ListOfAllPropertyUseTypes"
]
usage_column = next((col for col in candidate_columns if col in df.columns), None)

if usage_column is None:
    raise ValueError("No building type column found in the raw dataset.")

print("Using building type column:", usage_column)

filtered_df = df[df[usage_column].isin(non_residential_keywords)]
print("Filtered rows:", filtered_df.shape[0], "/", df.shape[0])
print("Retained share:", filtered_df.shape[0] / len(df))

In [ ]:
purge_path = "data/2016_Building_Energy_Benchmarking_Purge.csv"
filtered_df.to_csv(purge_path, index=False)
print("Saved filtered dataset to:", purge_path)

## Data overview and missing values

Examine the data types and detect columns with missing values before cleaning.

In [ ]:
display(filtered_df.head())
print("Data types and non-null counts:")
filtered_df.info()

In [ ]:
missing_data = filtered_df.isnull().sum().to_frame('missing_count')
missing_data['missing_pct'] = 100 * missing_data['missing_count'] / len(filtered_df)
missing_data = missing_data[missing_data['missing_count'] > 0].sort_values('missing_pct', ascending=False)
display(missing_data)

## Categorical variable review

Inspect the number of unique values in string columns to understand which features are suitable for encoding.

In [ ]:
categorical_columns = filtered_df.select_dtypes(include=['object', 'category']).columns.tolist()
categorical_counts = filtered_df[categorical_columns].nunique().sort_values()
display(categorical_counts.to_frame('unique_values'))

## Semantic review of key columns

This table summarizes the main column groups and their likely role in the modeling pipeline.

| Column | Role | Notes |
| --- | --- | --- |
| `SiteEnergyUse(kBtu)` | Target variable | Main energy consumption target for regression. |
| `PropertyGFATotal` | Building size | Strong predictor of energy use. |
| `YearBuilt` | Building age | Useful as a physical descriptor. |
| `PrimaryPropertyType` | Building usage | Strong semantic feature. |
| `LargestPropertyUseType` | Usage category | May overlap with `PrimaryPropertyType` but still useful. |
| `ComplianceStatus` | Data quality filter | Use it to keep only reliable records. |
| `TotalGHGEmissions` | Emissions outcome | Related to energy use and can support domain analysis. |

## Compliance status analysis

Review the distribution of compliance statuses to decide whether to restrict the dataset to records with reliable reporting.

In [ ]:
if 'ComplianceStatus' in filtered_df.columns:
    counts = filtered_df['ComplianceStatus'].value_counts(dropna=False)
    pct = filtered_df['ComplianceStatus'].value_counts(normalize=True, dropna=False) * 100
    compliance_summary = pd.DataFrame({
        'count': counts,
        'percent': pct.round(2)
    })
    display(compliance_summary)
else:
    print('ComplianceStatus column not found in this dataset.')

## Summary and next step

This exploratory notebook is meant to give students a first look at the raw dataset structure and the main semantic groups. The next notebook, `01_clean_data.ipynb`, uses this understanding to perform data cleaning, deduplication, and outlier detection before feature engineering.